# Group DRO — Waterbirds

Reproduction des résultats du papier **"Distributionally Robust Neural Networks for Group Shifts"** (Sagawa et al., 2020) sur le dataset Waterbirds.

**Problème** : ERM apprend une corrélation spurieuse fond ↔ classe (waterbird sur eau, landbird sur terre). Le worst-group accuracy s'effondre.

**Solution** : GroupDRO minimise la loss du **pire groupe** plutôt que la moyenne.

**4 groupes** :
| Groupe | Label | Fond | Train n |
|--------|-------|------|---------|
| 0 | Landbird | Terre | ~3498 (majoritaire) |
| 1 | Landbird | Eau | ~184 (minoritaire) |
| 2 | Waterbird | Terre | ~56 (minoritaire) |
| 3 | Waterbird | Eau | ~1057 (majoritaire) |

In [ ]:
# Cell 1 — Imports & device
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"{torch.__version__=}")
print(f"Using {device=}")

In [ ]:
# Cell 2 — Mount Google Drive (pour sauvegarder les checkpoints)
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = "/content/drive/MyDrive/group_dro_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

In [ ]:
# Cell 3 — Download Waterbirds
!wget -q https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz
!tar -xzf waterbird_complete95_forest2water2.tar.gz

DATA_DIR = "/content/waterbird_complete95_forest2water2"
print("Files:", os.listdir(DATA_DIR)[:5], "...")
df = pd.read_csv(os.path.join(DATA_DIR, 'metadata.csv'))
print(f"Total images: {len(df)}")
print(df[['y', 'place', 'split']].value_counts().sort_index())

In [ ]:
# Cell 4 — Dataset Waterbirds
#
# Chaque sample retourne (image, y, group)
# y      : 0 = landbird, 1 = waterbird
# place  : 0 = land background, 1 = water background
# group  : y * 2 + place  → {0, 1, 2, 3}

GROUP_NAMES = [
    'Landbird / Land bg',
    'Landbird / Water bg',
    'Waterbird / Land bg',
    'Waterbird / Water bg',
]
CLASS_NAMES = ['Landbird', 'Waterbird']

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_transform = transforms.Compose([
    transforms.Resize((int(224 * 256/224), int(224 * 256/224))),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class WaterbirdsDataset(Dataset):
    def __init__(self, data_dir, split, transform=None):
        """
        split: 'train' (0), 'val' (1), 'test' (2)
        """
        split_map = {'train': 0, 'val': 1, 'test': 2}
        self.data_dir = data_dir
        self.transform = transform

        df = pd.read_csv(os.path.join(data_dir, 'metadata.csv'))
        df = df[df['split'] == split_map[split]].reset_index(drop=True)

        self.filenames = df['img_filename'].values
        self.y = torch.tensor(df['y'].values, dtype=torch.long)
        self.place = torch.tensor(df['place'].values, dtype=torch.long)
        self.group = self.y * 2 + self.place  # 4 groupes
        self.n_groups = 4
        self.n_classes = 2

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.data_dir, self.filenames[idx])).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.y[idx], self.group[idx]

    def group_counts(self):
        return torch.bincount(self.group, minlength=self.n_groups).float()

    def group_str(self, g):
        return GROUP_NAMES[g]


train_ds = WaterbirdsDataset(DATA_DIR, 'train', transform=train_transform)
val_ds   = WaterbirdsDataset(DATA_DIR, 'val',   transform=eval_transform)
test_ds  = WaterbirdsDataset(DATA_DIR, 'test',  transform=eval_transform)

print("=== Distribution des groupes ===")
for split_name, ds in [('Train', train_ds), ('Val', val_ds), ('Test', test_ds)]:
    counts = ds.group_counts()
    print(f"\n{split_name} ({len(ds)} images):")
    for g, n in enumerate(counts):
        print(f"  Groupe {g} [{GROUP_NAMES[g]}]: {int(n)} ({100*n/len(ds):.1f}%)")

In [ ]:
# Cell 5 — Visualisation des 4 groupes
denorm = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
shown = {g: False for g in range(4)}

for idx in range(len(train_ds)):
    img, y, g = train_ds[idx]
    g = g.item()
    if not shown[g]:
        img_show = denorm(img).permute(1, 2, 0).numpy().clip(0, 1)
        axes[g].imshow(img_show)
        axes[g].set_title(GROUP_NAMES[g], fontsize=9)
        axes[g].axis('off')
        shown[g] = True
    if all(shown.values()):
        break

plt.suptitle('Exemples des 4 groupes Waterbirds', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 — DataLoaders
BATCH_SIZE = 128

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} images, {len(train_loader)} batches")
print(f"Val:   {len(val_ds)} images")
print(f"Test:  {len(test_ds)} images")

In [ ]:
# Cell 7 — Modèle ResNet-50 (2 classes)
def get_model():
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model.to(device)

print("Model factory défini.")

In [ ]:
# Cell 8 — LossComputer (GroupDRO)
#
# Adapté de group_DRO/loss.py (Sagawa et al., 2020).
# Clé : `adv_probs` — vecteur de poids par groupe mis à jour à chaque batch.
# Plus un groupe a une loss élevée, plus son poids augmente → le modèle
# est forcé de réduire la loss du pire groupe.

class LossComputer:
    def __init__(self, criterion, is_robust, dataset, gamma=0.1, step_size=0.01, adj=None):
        self.criterion   = criterion
        self.is_robust   = is_robust
        self.gamma       = gamma
        self.step_size   = step_size
        self.n_groups    = dataset.n_groups
        self.group_str   = dataset.group_str

        self.group_counts = dataset.group_counts().to(device)
        self.group_frac   = self.group_counts / self.group_counts.sum()

        self.adj = torch.zeros(self.n_groups).to(device) if adj is None else torch.tensor(adj).float().to(device)

        # Poids adversariaux initialisés uniformément
        self.adv_probs          = torch.ones(self.n_groups).to(device) / self.n_groups
        self.exp_avg_loss       = torch.zeros(self.n_groups).to(device)
        self.exp_avg_initialized = torch.zeros(self.n_groups).bool().to(device)

        self.reset_stats()

    def loss(self, logits, y, g, is_training):
        per_sample_loss = self.criterion(logits, y)  # (B,)

        # Loss et accuracy moyennes par groupe
        group_map   = (g.unsqueeze(0) == torch.arange(self.n_groups).unsqueeze(1).to(device)).float()
        group_count = group_map.sum(1)
        denom       = group_count + (group_count == 0).float()
        group_loss  = (group_map @ per_sample_loss) / denom
        group_acc   = (group_map @ (logits.argmax(1) == y).float()) / denom

        # Mise à jour de la moyenne exponentielle des losses par groupe
        prev_w = (1 - self.gamma * (group_count > 0).float()) * self.exp_avg_initialized.float()
        self.exp_avg_loss        = self.exp_avg_loss * prev_w + group_loss * (1 - prev_w)
        self.exp_avg_initialized = self.exp_avg_initialized | (group_count > 0)

        if self.is_robust:
            # Mise à jour des poids adversariaux (exponentiated gradient)
            adjusted_loss    = group_loss + self.adj / torch.sqrt(self.group_counts)
            self.adv_probs   = self.adv_probs * torch.exp(self.step_size * adjusted_loss.detach())
            self.adv_probs   = self.adv_probs / self.adv_probs.sum()
            actual_loss      = group_loss @ self.adv_probs
        else:
            actual_loss = per_sample_loss.mean()

        # Stats
        denom2 = self.processed_data_counts + group_count
        denom2 += (denom2 == 0).float()
        pw, cw = self.processed_data_counts / denom2, group_count / denom2
        self.avg_group_loss = pw * self.avg_group_loss + cw * group_loss
        self.avg_group_acc  = pw * self.avg_group_acc  + cw * group_acc
        self.avg_actual_loss = (self.batch_count / (self.batch_count + 1)) * self.avg_actual_loss \
                             + (1 / (self.batch_count + 1)) * actual_loss.item()
        self.processed_data_counts += group_count
        self.batch_count += 1

        return actual_loss

    def reset_stats(self):
        self.processed_data_counts = torch.zeros(self.n_groups).to(device)
        self.avg_group_loss        = torch.zeros(self.n_groups).to(device)
        self.avg_group_acc         = torch.zeros(self.n_groups).to(device)
        self.avg_actual_loss       = 0.
        self.batch_count           = 0

    def get_group_acc(self):
        return self.avg_group_acc.cpu().numpy()

    def worst_group_acc(self):
        return self.avg_group_acc.min().item()

    def avg_acc(self):
        frac = self.processed_data_counts / self.processed_data_counts.sum()
        return (frac @ self.avg_group_acc).item()

print("LossComputer défini.")

In [ ]:
# Cell 9 — Fonctions train / evaluate

def run_epoch(model, loader, loss_computer, optimizer=None, is_training=True):
    model.train() if is_training else model.eval()
    loss_computer.reset_stats()

    with torch.set_grad_enabled(is_training):
        for x, y, g in loader:
            x, y, g = x.to(device), y.to(device), g.to(device)
            logits = model(x)
            loss   = loss_computer.loss(logits, y, g, is_training)
            if is_training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

    return {
        'avg_acc':        loss_computer.avg_acc(),
        'worst_acc':      loss_computer.worst_group_acc(),
        'group_acc':      loss_computer.get_group_acc(),
        'avg_loss':       loss_computer.avg_actual_loss,
    }


def train_model(name, is_robust, n_epochs, lr, weight_decay, gamma=0.1, step_size=0.01):
    print(f"\n{'='*55}")
    print(f"  {name} | {'GroupDRO' if is_robust else 'ERM'} | "
          f"lr={lr} | wd={weight_decay} | epochs={n_epochs}")
    print(f"{'='*55}")

    model     = get_model()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss(reduction='none')

    train_lc = LossComputer(criterion, is_robust=is_robust, dataset=train_ds,
                            gamma=gamma, step_size=step_size)
    val_lc   = LossComputer(criterion, is_robust=False, dataset=val_ds)

    history = []
    best_worst_acc = 0
    best_state = None

    for epoch in range(1, n_epochs + 1):
        t0 = time.time()
        train_stats = run_epoch(model, train_loader, train_lc, optimizer, is_training=True)
        val_stats   = run_epoch(model, val_loader,   val_lc,   is_training=False)
        elapsed = time.time() - t0

        # Sauvegarder le meilleur modèle selon la worst-group accuracy sur val
        if val_stats['worst_acc'] > best_worst_acc:
            best_worst_acc = val_stats['worst_acc']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        history.append({'epoch': epoch, **{f'train_{k}': v for k, v in train_stats.items()},
                                          **{f'val_{k}': v for k, v in val_stats.items()}})

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{n_epochs} | "
                  f"train_loss={train_stats['avg_loss']:.3f} | "
                  f"val_avg={val_stats['avg_acc']*100:.1f}% | "
                  f"val_worst={val_stats['worst_acc']*100:.1f}% | "
                  f"{elapsed:.0f}s")

    # Charger le meilleur modèle et évaluer sur test
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), os.path.join(CKPT_DIR, f"{name.replace(' ', '_')}.pth"))

    test_lc   = LossComputer(criterion, is_robust=False, dataset=test_ds)
    test_stats = run_epoch(model, test_loader, test_lc, is_training=False)

    print(f"\n--- Test results ({name}) ---")
    for g, acc in enumerate(test_stats['group_acc']):
        print(f"  {GROUP_NAMES[g]:25s}: {acc*100:.1f}%")
    print(f"  {'Average':25s}: {test_stats['avg_acc']*100:.1f}%")
    print(f"  {'Worst-group':25s}: {test_stats['worst_acc']*100:.1f}%")

    return history, test_stats

print("Fonctions d'entraînement définies.")

In [ ]:
# Cell 10 — ERM (baseline)
# Hyperparamètres du papier (Table 5, Waterbirds ERM)
# lr=0.001, weight_decay=1e-4, 300 epochs
# Résultats attendus : avg ~97%, worst-group ~72%

history_erm, test_erm = train_model(
    name='ERM',
    is_robust=False,
    n_epochs=300,
    lr=0.001,
    weight_decay=1e-4,
)

In [ ]:
# Cell 11 — GroupDRO
# Hyperparamètres du papier (Table 5, Waterbirds GroupDRO)
# lr=0.001, weight_decay=1e-1 (régularisation forte !), 300 epochs
# gamma=0.1 (step size mise à jour poids adversariaux)
# Résultats attendus : avg ~93%, worst-group ~91%

history_dro, test_dro = train_model(
    name='GroupDRO',
    is_robust=True,
    n_epochs=300,
    lr=0.001,
    weight_decay=1e-1,   # clé du papier : régularisation forte
    gamma=0.1,
    step_size=0.01,
)

In [ ]:
# Cell 12 — Comparaison ERM vs GroupDRO (résultats du papier inclus)

# Résultats du papier (Table 1)
paper_erm = {'group_acc': [0.987, 0.564, 0.721, 0.967], 'avg_acc': 0.973, 'worst_acc': 0.564}
paper_dro = {'group_acc': [0.933, 0.884, 0.892, 0.957], 'avg_acc': 0.935, 'worst_acc': 0.884}

print(f"{'Groupe':25s} | {'ERM (papier)':>12s} | {'ERM (repro)':>12s} | {'DRO (papier)':>12s} | {'DRO (repro)':>12s}")
print("-" * 80)
for g in range(4):
    print(f"{GROUP_NAMES[g]:25s} | "
          f"{paper_erm['group_acc'][g]*100:>11.1f}% | "
          f"{test_erm['group_acc'][g]*100:>11.1f}% | "
          f"{paper_dro['group_acc'][g]*100:>11.1f}% | "
          f"{test_dro['group_acc'][g]*100:>11.1f}%")
print("-" * 80)
for key, label in [('avg_acc', 'Average'), ('worst_acc', 'Worst-group')]:
    print(f"{label:25s} | "
          f"{paper_erm[key]*100:>11.1f}% | "
          f"{test_erm[key]*100:>11.1f}% | "
          f"{paper_dro[key]*100:>11.1f}% | "
          f"{test_dro[key]*100:>11.1f}%")

In [ ]:
# Cell 13 — Visualisations

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Plot 1 : Worst-group accuracy au fil des epochs ---
epochs = [h['epoch'] for h in history_erm]
axes[0].plot(epochs, [h['val_worst_acc']*100 for h in history_erm], label='ERM', color='steelblue')
axes[0].plot(epochs, [h['val_worst_acc']*100 for h in history_dro], label='GroupDRO', color='coral')
axes[0].axhline(paper_erm['worst_acc']*100, color='steelblue', linestyle='--', alpha=0.5, label='ERM (papier)')
axes[0].axhline(paper_dro['worst_acc']*100, color='coral',     linestyle='--', alpha=0.5, label='DRO (papier)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Worst-group Accuracy (%)')
axes[0].set_title('Worst-group Accuracy (val)')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# --- Plot 2 : Average accuracy au fil des epochs ---
axes[1].plot(epochs, [h['val_avg_acc']*100 for h in history_erm], label='ERM', color='steelblue')
axes[1].plot(epochs, [h['val_avg_acc']*100 for h in history_dro], label='GroupDRO', color='coral')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Average Accuracy (%)')
axes[1].set_title('Average Accuracy (val)\n[ERM gagne en average, DRO gagne en worst]')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

# --- Plot 3 : Bar chart par groupe (test set) ---
x = np.arange(4)
w = 0.35
b1 = axes[2].bar(x - w/2, [test_erm['group_acc'][g]*100 for g in range(4)], w, label='ERM', color='steelblue')
b2 = axes[2].bar(x + w/2, [test_dro['group_acc'][g]*100 for g in range(4)], w, label='GroupDRO', color='coral')
axes[2].bar_label(b1, fmt='%.1f%%', padding=2, fontsize=8)
axes[2].bar_label(b2, fmt='%.1f%%', padding=2, fontsize=8)
axes[2].set_xticks(x)
axes[2].set_xticklabels(['G0\nLand/Land', 'G1\nLand/Water', 'G2\nWater/Land', 'G3\nWater/Water'], fontsize=8)
axes[2].set_ylim(0, 115)
axes[2].set_ylabel('Accuracy (%)')
axes[2].set_title('Accuracy par groupe (test)\n[G1 et G2 = groupes minoritaires]')
axes[2].legend()
axes[2].grid(alpha=0.3, axis='y')

plt.suptitle('ERM vs GroupDRO — Waterbirds', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, 'erm_vs_dro.png'), dpi=150)
plt.show()
print("Figure saved.")

In [ ]:
# Cell 14 — Ablation : impact du weight decay sur GroupDRO
#
# Le papier insiste : la régularisation forte est CRITIQUE pour GroupDRO.
# Sans elle, GroupDRO overfit sur le pire groupe et les résultats s'effondrent.
# On teste 3 valeurs pour le montrer empiriquement.

wd_values = [1e-4, 1e-2, 1e-1]   # faible / moyen / fort (valeur du papier)
ablation_results = {}

for wd in wd_values:
    _, test_stats = train_model(
        name=f'DRO_wd={wd}',
        is_robust=True,
        n_epochs=100,       # 100 epochs suffit pour voir la tendance
        lr=0.001,
        weight_decay=wd,
    )
    ablation_results[wd] = test_stats

# Résumé
print("\n=== Ablation Weight Decay (GroupDRO, 100 epochs) ===")
print(f"{'Weight Decay':>15s} | {'Avg Acc':>10s} | {'Worst-group Acc':>16s}")
print("-" * 48)
for wd, stats in ablation_results.items():
    print(f"{wd:>15.0e} | {stats['avg_acc']*100:>9.1f}% | {stats['worst_acc']*100:>15.1f}%")

In [ ]:
# Cell 15 — Visualisation ablation weight decay

fig, ax = plt.subplots(figsize=(7, 4))
wds = list(ablation_results.keys())
avg_accs   = [ablation_results[w]['avg_acc']*100   for w in wds]
worst_accs = [ablation_results[w]['worst_acc']*100 for w in wds]

x = np.arange(len(wds))
w = 0.35
b1 = ax.bar(x - w/2, avg_accs,   w, label='Average Accuracy', color='steelblue')
b2 = ax.bar(x + w/2, worst_accs, w, label='Worst-group Accuracy', color='coral')
ax.bar_label(b1, fmt='%.1f%%', padding=2, fontsize=9)
ax.bar_label(b2, fmt='%.1f%%', padding=2, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([f'wd={w}' for w in wds])
ax.set_ylim(0, 115)
ax.set_ylabel('Accuracy (%)')
ax.set_title('GroupDRO — Impact du Weight Decay\n(contribution principale du papier)')
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, 'ablation_weight_decay.png'), dpi=150)
plt.show()